# Run All Comparison Experiments (Local Machine)

This notebook runs the equivalent of `run_all_comparisons.sh` on your local machine.

## Experiments

**Three-Way Comparisons:**
1. **Baseline**: Full-volume, no lesion filtering
2. **Filtered Baseline**: Full-volume with lesion size filtering
3. **ROI-Cropped**: Adaptive crop/pad (tumor-centered volumes)

This local version currently runs the TabPFN comparisons only.

## Setup

In [1]:
from pathlib import Path
import sys
import torch

# Add project root to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print(f"Python: {sys.version}")

Project root: c:\Users\cahel\Desktop\Med3Tab-PFN
Python: 3.10.18 | packaged by conda-forge | (main, Jun  4 2025, 14:42:04) [MSC v.1943 64 bit (AMD64)]


In [2]:
# Check GPU/CPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nDevice: {device}")

if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"CUDA Version: {torch.version.cuda}")
else:
    print("⚠️  Running on CPU. Experiments will be slower.")


Device: cpu
⚠️  Running on CPU. Experiments will be slower.


In [3]:
# Setup SAM-Med3D model using medim (HuggingFace weights)
from pathlib import Path
import medim

# Define checkpoint path
CHECKPOINT_PATH = project_root / "SAM-Med3D-main" / "SAM-Med3D-main" / "ckpt" / "sam_med3d_turbo.pth"
ckpt_dir = CHECKPOINT_PATH.parent
ckpt_dir.mkdir(parents=True, exist_ok=True)

# Download checkpoint if not present
if not CHECKPOINT_PATH.exists():
    url = "https://huggingface.co/blueyo0/SAM-Med3D/resolve/main/sam_med3d_turbo.pth"
    print(f"Checkpoint missing. Downloading to: {CHECKPOINT_PATH}")
    try:
        import urllib.request, shutil
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as r, open(CHECKPOINT_PATH, 'wb') as f:
            shutil.copyfileobj(r, f)
        size_mb = CHECKPOINT_PATH.stat().st_size / (1024*1024)
        print(f"✅ Download complete: {size_mb:.1f} MB")
        if size_mb < 300:
            print("⚠️  File size is unexpectedly small. If loading fails, delete the file and rerun this cell.")
    except Exception as e:
        print(f"❌ Failed to download checkpoint: {e}")
        print("You can manually download from:")
        print(url)
else:
    size_mb = CHECKPOINT_PATH.stat().st_size / (1024*1024)
    print(f"✅ Checkpoint present: {CHECKPOINT_PATH} ({size_mb:.1f} MB)")

# Load SAM-Med3D model using medim (same approach as SAM_Visualization)
print("\n📦 Loading SAM-Med3D model via medim...")
sam_model = medim.create_model(
    "SAM-Med3D",
    pretrained=True,
    checkpoint_path=str(CHECKPOINT_PATH)
).to(device)
sam_model.eval()
print("✅ SAM-Med3D model loaded successfully via medim!")
print(f"   Model device: {next(sam_model.parameters()).device}")

✅ Checkpoint present: c:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth (383.5 MB)

📦 Loading SAM-Med3D model via medim...
creating model SAM-Med3D
try to load pretrained weights from c:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\ckpt\sam_med3d_turbo.pth
✅ SAM-Med3D model loaded successfully via medim!
   Model device: cpu


## Experiment 1: TabPFN Three-Way Comparison

Runs baseline, filtered baseline, and ROI-cropped experiments using TabPFN.

In [4]:
from med3pipe.pipelines import run_multi_tabpfn
from med3pipe.tabular.lesion_filter import LesionSizeFilter
import pandas as pd

# Configuration
config_path = project_root / "configs" / "datasets.yaml"
results_dir_tabpfn = project_root / "results" / "three_way_comparison"
results_dir_tabpfn.mkdir(parents=True, exist_ok=True)

print("="*70)
print("EXPERIMENT 1: TabPFN Three-Way Comparison")
print("="*70)
print(f"Results will be saved to: {results_dir_tabpfn}")

EXPERIMENT 1: TabPFN Three-Way Comparison
Results will be saved to: c:\Users\cahel\Desktop\Med3Tab-PFN\results\three_way_comparison


### 1a. Baseline (Full-Volume, No Filtering)

In [5]:
print("\n" + "="*70)
print("1a. BASELINE (Full-Volume, No Filtering)")
print("="*70 + "\n")

try:
    results_baseline = run_multi_tabpfn(
        config_path=config_path,
        
        # Full-volume preprocessing, NO filtering
        use_roi_crop=False,
        
        # Model parameters - use pre-loaded medim model
        model=sam_model,  # Pre-loaded SAM-Med3D model via medim
        device=device,
        
        # Feature extraction
        skip_existing_embeddings=False,  # Reuse if available
        
        # TabPFN parameters
        n_components_max=500,
        random_state=42,
        n_splits=5,
        
        # Output
        outputs_base_dir=results_dir_tabpfn / "baseline",
        save_summary=True,
        summary_path=results_dir_tabpfn / "baseline_summary.csv",
    )
    
    print(f"\n✅ Baseline experiment completed")
    print(f"   Results: {results_dir_tabpfn / 'baseline_summary.csv'}")
    
except Exception as e:
    print(f"\n❌ Baseline experiment failed: {e}")
    import traceback
    traceback.print_exc()


1a. BASELINE (Full-Volume, No Filtering)


✅ gist: Reusing existing embeddings...
Prepared 25 cases ...
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
✅ Using pre-loaded model (e.g., via medim)
To extract: 246 from C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\dat

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Fold 1 - Accuracy: 0.6400, F1: 0.6305

--- Fold 2/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (197, 384) (samples x features)
  Val shape:   (49, 384) (samples x features)

Fold 2 - Accuracy: 0.5510, F1: 0.5493

--- Fold 3/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (197, 384) (samples x features)
  Val shape:   (49, 384) (samples x features)

Fold 2 - Accuracy: 0.5510, F1: 0.5493

--- Fold 3/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (197, 384) (samples x features)
  Val shape:   (49, 384) (samples x features)

Fold 3 - Accuracy: 0.5510, F1: 0.5508

--- Fold 4/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (197, 384) (samples x features)
  Val shape:   (49, 384) (samples x features)

Fold 3 - Accuracy: 0.5510, F1: 0.5508

--- Fold 4/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (197, 384) (samples x features)
  Val shape:   (49, 384) (samples x features)

Fold 4 - Accuracy: 0.6531, F1: 0.6507

--- Fo

### 1b. Filtered Baseline (Full-Volume + Lesion Filtering)

In [6]:
print("\n" + "="*70)
print("1b. FILTERED BASELINE (Full-Volume + Lesion Filtering)")
print("="*70 + "\n")

# Lesion filter configuration
lesion_filter = LesionSizeFilter(
    min_voxels=300,      # Minimum 100 voxels
    min_dimension=5,     # At least 3 voxels in each dimension
    min_density=0.1,     # At least 10% of bounding box should be lesion
)

print(f"Lesion filter: min_voxels={lesion_filter.min_voxels}, "
      f"min_dimension={lesion_filter.min_dimension}, min_density={lesion_filter.min_density}\n")

try:
    results_filtered = run_multi_tabpfn(
        config_path=config_path,
        
        # Full-volume preprocessing with lesion filtering
        use_roi_crop=False,
        min_voxels=lesion_filter.min_voxels,
        min_dimension=lesion_filter.min_dimension,
        min_density=lesion_filter.min_density,
        
        # Model parameters - use pre-loaded medim model
        model=sam_model,  # Pre-loaded SAM-Med3D model via medim
        device=device,
        
        # Feature extraction
        skip_existing_embeddings=False,  # Reuse if available
        
        # TabPFN parameters
        n_components_max=500,
        random_state=42,
        n_splits=5,
        
        # Output
        outputs_base_dir=results_dir_tabpfn / "filtered_baseline",
        save_summary=True,
        summary_path=results_dir_tabpfn / "filtered_baseline_summary.csv",
    )
    
    print(f"\n✅ Filtered baseline experiment completed")
    print(f"   Results: {results_dir_tabpfn / 'filtered_baseline_summary.csv'}")
    
except Exception as e:
    print(f"\n❌ Filtered baseline experiment failed: {e}")
    import traceback
    traceback.print_exc()


1b. FILTERED BASELINE (Full-Volume + Lesion Filtering)

Lesion filter: min_voxels=300, min_dimension=5, min_density=0.1


✅ gist: Reusing existing embeddings...
Prepared 25 cases ...
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 200 cases ...
Prepared 225 cases ...
Prepared 225 cases ...
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Done. Prepared 246 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Validation set copied to: C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\validation\gist\ct_GIST | Train: 196 | Val: 50
✅ Using pre-loaded model (e.g., via medim)
To extract

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fold 1 - Accuracy: 0.5909, F1: 0.3714

--- Fold 2/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (86, 384) (samples x features)
  Val shape:   (22, 384) (samples x features)

Fold 2 - Accuracy: 0.6364, F1: 0.4824

--- Fold 3/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (86, 384) (samples x features)
  Val shape:   (22, 384) (samples x features)

Fold 2 - Accuracy: 0.6364, F1: 0.4824

--- Fold 3/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (86, 384) (samples x features)
  Val shape:   (22, 384) (samples x features)

Fold 3 - Accuracy: 0.5455, F1: 0.4271

--- Fold 4/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (87, 384) (samples x features)
  Val shape:   (21, 384) (samples x features)

Fold 3 - Accuracy: 0.5455, F1: 0.4271

--- Fold 4/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (87, 384) (samples x features)
  Val shape:   (21, 384) (samples x features)



c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fold 4 - Accuracy: 0.5714, F1: 0.3636

--- Fold 5/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (87, 384) (samples x features)
  Val shape:   (21, 384) (samples x features)

Fold 5 - Accuracy: 0.6190, F1: 0.5962

K-Fold Cross-Validation Summary
Accuracy:  0.5926 ± 0.0325
F1 Score:  0.4481 ± 0.0855
ROC AUC:   0.6980 ± 0.0918


✅ lipo: Reusing existing embeddings...
Fold 5 - Accuracy: 0.6190, F1: 0.5962

K-Fold Cross-Validation Summary
Accuracy:  0.5926 ± 0.0325
F1 Score:  0.4481 ± 0.0855
ROC AUC:   0.6980 ± 0.0918


✅ lipo: Reusing existing embeddings...
Prepared 25 cases ...
Prepared 25 cases ...
Prepared 50 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 100 cases ...
Done. Prepared 115 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\lipo\ct_LIPO
Done. Prepared 115 cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\lipo\ct_LIPO
Validation s

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fold 1 - Accuracy: 0.5294, F1: 0.3462

--- Fold 2/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (65, 384) (samples x features)
  Val shape:   (16, 384) (samples x features)



c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Fold 2 - Accuracy: 0.5625, F1: 0.3600

--- Fold 3/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (65, 384) (samples x features)
  Val shape:   (16, 384) (samples x features)

Fold 3 - Accuracy: 0.5625, F1: 0.4589

--- Fold 4/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (65, 384) (samples x features)
  Val shape:   (16, 384) (samples x features)

Fold 3 - Accuracy: 0.5625, F1: 0.4589

--- Fold 4/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (65, 384) (samples x features)
  Val shape:   (16, 384) (samples x features)

Fold 4 - Accuracy: 0.4375, F1: 0.4170

--- Fold 5/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (65, 384) (samples x features)
  Val shape:   (16, 384) (samples x features)

Fold 4 - Accuracy: 0.4375, F1: 0.4170

--- Fold 5/5 ---

ENCODER OUTPUT DIMENSIONS (before PCA):
  Train shape: (65, 384) (samples x features)
  Val shape:   (16, 384) (samples x features)

Fold 5 - Accuracy: 0.5000, F1: 0.3333

K-Fold Cros

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


### 1c. ROI-Cropped (Adaptive Crop/Pad)

In [7]:
print("\n" + "="*70)
print("1c. ROI-CROPPED (Adaptive Crop/Pad)")
print("="*70 + "\n")

try:
    results_roi = run_multi_tabpfn(
        config_path=config_path,
        
        # ROI-centric preprocessing (proposed)
        use_roi_crop=True,
        roi_margin=30,          # 10 voxels context
        roi_target_size=128,    # Final volume size
        
        # Model parameters - use pre-loaded medim model
        model=sam_model,  # Pre-loaded SAM-Med3D model via medim
        device=device,
        
        # Feature extraction (force re-extraction for ROI data)
        skip_existing_embeddings=True,
        
        # TabPFN parameters
        n_components_max=500,
        random_state=42,
        n_splits=5,
        
        # Output
        outputs_base_dir=results_dir_tabpfn / "roi_cropped",
        save_summary=True,
        summary_path=results_dir_tabpfn / "roi_cropped_summary.csv",
    )
    
    print(f"\n✅ ROI-cropped experiment completed")
    print(f"   Results: {results_dir_tabpfn / 'roi_cropped_summary.csv'}")
    
except Exception as e:
    print(f"\n❌ ROI-cropped experiment failed: {e}")
    import traceback
    traceback.print_exc()


1c. ROI-CROPPED (Adaptive Crop/Pad)


🔄 gist: Skipping existing embeddings. Re-extracting...
Prepared 25 ROI-cropped cases ...
Prepared 25 ROI-cropped cases ...
Prepared 50 ROI-cropped cases ...
Prepared 50 ROI-cropped cases ...
Prepared 75 ROI-cropped cases ...
Prepared 75 ROI-cropped cases ...
Prepared 100 ROI-cropped cases ...
Prepared 100 ROI-cropped cases ...
Prepared 125 ROI-cropped cases ...
Prepared 125 ROI-cropped cases ...
Prepared 150 ROI-cropped cases ...
Prepared 150 ROI-cropped cases ...
Prepared 175 ROI-cropped cases ...
Prepared 175 ROI-cropped cases ...
Prepared 200 ROI-cropped cases ...
Prepared 200 ROI-cropped cases ...
Prepared 225 ROI-cropped cases ...
Prepared 225 ROI-cropped cases ...
Done. Prepared 246 ROI-cropped cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Done. Prepared 246 ROI-cropped cases to C:\Users\cahel\Desktop\Med3Tab-PFN\SAM-Med3D-main\SAM-Med3D-main\data\train\gist\ct_GIST
Validation set copied to

### 1d. Compare TabPFN Results

In [8]:
print("\n" + "="*70)
print("TABPFN COMPARISON")
print("="*70 + "\n")

try:
    # Load all three results
    df_baseline = pd.read_csv(results_dir_tabpfn / "baseline_summary.csv")
    df_filtered = pd.read_csv(results_dir_tabpfn / "filtered_baseline_summary.csv")
    df_roi = pd.read_csv(results_dir_tabpfn / "roi_cropped_summary.csv")
    
    # Merge on dataset
    comparison = pd.merge(
        df_baseline[['dataset', 'accuracy', 'macro_f1', 'roc_auc']],
        df_filtered[['dataset', 'accuracy', 'macro_f1', 'roc_auc']],
        on='dataset',
        suffixes=('_baseline', '_filtered')
    )
    comparison = pd.merge(
        comparison,
        df_roi[['dataset', 'accuracy', 'macro_f1', 'roc_auc']],
        on='dataset'
    )
    comparison.rename(columns={
        'accuracy': 'accuracy_roi',
        'macro_f1': 'macro_f1_roi',
        'roc_auc': 'roc_auc_roi'
    }, inplace=True)
    
    # Calculate improvements (vs baseline)
    comparison['accuracy_delta_filtered'] = comparison['accuracy_filtered'] - comparison['accuracy_baseline']
    comparison['accuracy_delta_roi'] = comparison['accuracy_roi'] - comparison['accuracy_baseline']
    comparison['f1_delta_filtered'] = comparison['macro_f1_filtered'] - comparison['macro_f1_baseline']
    comparison['f1_delta_roi'] = comparison['macro_f1_roi'] - comparison['macro_f1_baseline']
    comparison['auc_delta_filtered'] = comparison['roc_auc_filtered'] - comparison['roc_auc_baseline']
    comparison['auc_delta_roi'] = comparison['roc_auc_roi'] - comparison['roc_auc_baseline']
    
    # Save comparison
    comparison_path = results_dir_tabpfn / "three_way_comparison.csv"
    comparison.to_csv(comparison_path, index=False)
    
    print(comparison.to_string(index=False))
    print(f"\n✅ Comparison saved to: {comparison_path}")
    
    # Summary statistics
    print("\n" + "-"*70)
    print("AVERAGE IMPROVEMENTS vs. BASELINE")
    print("-"*70)
    print("Filtered Baseline:")
    print(f"  Accuracy: {comparison['accuracy_delta_filtered'].mean():+.4f}")
    print(f"  F1 Score: {comparison['f1_delta_filtered'].mean():+.4f}")
    print(f"  ROC AUC:  {comparison['auc_delta_filtered'].mean():+.4f}")
    print("\nROI-Cropped:")
    print(f"  Accuracy: {comparison['accuracy_delta_roi'].mean():+.4f}")
    print(f"  F1 Score: {comparison['f1_delta_roi'].mean():+.4f}")
    print(f"  ROC AUC:  {comparison['auc_delta_roi'].mean():+.4f}")
    
except Exception as e:
    print(f"⚠️  Could not generate comparison: {e}")
    import traceback
    traceback.print_exc()


TABPFN COMPARISON

dataset  accuracy_baseline  macro_f1_baseline  roc_auc_baseline  accuracy_filtered  macro_f1_filtered  roc_auc_filtered  accuracy_roi  macro_f1_roi  roc_auc_roi  accuracy_delta_filtered  accuracy_delta_roi  f1_delta_filtered  f1_delta_roi  auc_delta_filtered  auc_delta_roi
   gist           0.605551           0.602788          0.643520           0.592641           0.448131          0.698006      0.576816      0.565366     0.606040                -0.012910           -0.028735          -0.154657     -0.037422            0.054486      -0.037480
   lipo           0.504348           0.485957          0.531818           0.518382           0.383086          0.515129      0.460870      0.449251     0.439394                 0.014035           -0.043478          -0.102871     -0.036706           -0.016689      -0.092424

✅ Comparison saved to: c:\Users\cahel\Desktop\Med3Tab-PFN\results\three_way_comparison\three_way_comparison.csv

--------------------------------------------

## Summary

All TabPFN experiments are complete. Results have been saved to:
- `results/three_way_comparison/baseline_summary.csv`
- `results/three_way_comparison/filtered_baseline_summary.csv`
- `results/three_way_comparison/roi_cropped_summary.csv`
- `results/three_way_comparison/three_way_comparison.csv` (aggregated comparison)

Review the CSVs above or reuse the final comparison dataframe printed in the previous cell for downstream analysis.